In [21]:
# ============================================================
# 0_FNS
# ============================================================

import os
from reportlab.pdfgen import canvas
from reportlab.lib.pagesizes import letter
from reportlab.lib.units import inch
from reportlab.lib.utils import ImageReader
from reportlab.lib import colors
from reportlab.pdfbase import pdfmetrics


# ------------------------------######------------------------------
# ############### CORE IMPORTABLE FUNCTION #########################
# ------------------------------######------------------------------
def _pdf_2201_nmhouse_GET_letter_proposal(
    bg_png_path,
    out_pdf_path,
    title="NEU MAGIC HOUSE",
    subtitle="\n\nA curated Thursday series by YerikoDJ",
):
    """
    Creates a 1-page LETTER PDF (8.5x11) using bg_png_path as full-page background,
    then overlays the Neu Magic House proposal text on top.

    Inputs:
      - bg_png_path: path to your washed/red letter background PNG
      - out_pdf_path: output PDF path

    Output:
      - returns out_pdf_path
    """

    # --------------------------
    # helpers
    # --------------------------
    def _wrap_lines(text, font_name, font_size, max_width):
        lines_out = []
        for para in (text or "").split("\n"):
            if para.strip() == "":
                lines_out.append("")
                continue

            words = para.split()
            line = ""
            for w in words:
                test = (line + " " + w).strip()
                if pdfmetrics.stringWidth(test, font_name, font_size) <= max_width:
                    line = test
                else:
                    if line:
                        lines_out.append(line)

                    # handle extremely long single words
                    if pdfmetrics.stringWidth(w, font_name, font_size) <= max_width:
                        line = w
                    else:
                        chunk = ""
                        for ch in w:
                            test2 = chunk + ch
                            if pdfmetrics.stringWidth(test2, font_name, font_size) <= max_width:
                                chunk = test2
                            else:
                                lines_out.append(chunk)
                                chunk = ch
                        line = chunk

            if line:
                lines_out.append(line)

        return lines_out

    def _draw_block(c, text, x, y, maxw, font, size, leading, space_after):
        c.setFont(font, size)
        lines = _wrap_lines(text, font, size, maxw)
        for ln in lines:
            if ln == "":
                y -= leading * 0.65
                continue
            c.drawString(x, y, ln)
            y -= leading
        y -= space_after
        return y

    # --------------------------
    # setup page
    # --------------------------
    W, H = letter
    c = canvas.Canvas(out_pdf_path, pagesize=letter)

    # background image full-page
    bg_img = ImageReader(bg_png_path)
    c.drawImage(bg_img, 0, 0, width=W, height=H, mask="auto")

    # text color (cream/off-white)
    cream = colors.Color(1, 0.97, 0.90)
    c.setFillColor(cream)

    # layout
    left = 0.75 * inch
    right = 0.65 * inch
    top = 1.05 * inch
    x = left
    y = H - top
    maxw = W - left - right

    # --------------------------
    # content (EDIT HERE)
    # --------------------------
    intro_1 = (
        "\nNeu Magic House is a curated series on selected Thursdays throughout the year, focused on House music."
    )

    intro_2 = (
        "Each night is built by starting with the right people. Reaching out personally is a fixed step in my calendar before every event."
    )

    collab_title = "CURATION"
    collab_body = (
        "Anyone can collect music. Not everyone knows what to play, when to play it, and why. — Jeff Mills. "
        "Emotion comes into curation, ensuring every set contributes to a larger movement. "
        "I can’t tell people what to play; I have to trust DJs. Neu Magic House is a door for them to understand the angle. "
        "As music curation merges visual aspects with space, I am creating an angle to help guide them. "
        "After having multiple residencies around Detroit, I understand the need to be consistent and intentional. "
        "The musical intention of this proposal is to explore House music. "
        "Steven Reaume has been a good mentor when it comes to bringing people together — the right people."
    )


    collab_list = (
        
    )

    identity_title = "VISUAL IDENTITY"
    identity_list = (
        "- Colors: black + orange-red + white\n"
        "- Stencil-style visuals\n"
        "- Nothing AI-looking"
    )

    calendar_title = "EVENT CALENDAR"
    calendar = [
        ("Feb 19", "Soojin b2b Marina · Live vocals: Vanessa Cuccia · Ashton b2b Yeriko"),
        ("Mar 12", "Yeriko · Charles Trees · Melo · DJ Cent"),
        ("Apr 9" ,  "TBA"),
        ("May 14", "Spike Before Movement · Yeriko · Harlow · Ryan Spencer + WSG"),
        ("Jun 11", "Summer Series (2 Stages) · Outside: YerikoDJ b2b Ashton · Inside: Erika Erie · ATM"),
        ("Jun 25", "Soojin Birthday : Full Female Line-Up · Inside & Outside"),
        ("Jul 16", "TBA - Global House  "),
        ("Aug 13", "inside Special Headliner (TBA) · Local DJs Outside"),
        ("Sep 17", "Yeriko · Duck · Mexico City Line-Up"),
        ("Oct 15", "TBA"),
        ("Nov 12", "YerikoDJ Album Release · Special Guests"),
        ("Dec 17", "Tropical Freeze ·Deluxe Tech House Tulum Vibez "),
    ]


    # --------------------------
    # draw text
    # --------------------------
    # Title
    y = _draw_block(c, title, x, y, maxw, "Helvetica-Bold", 30, 30 * 1.15, 2)
    y = _draw_block(c, subtitle, x, y, maxw, "Helvetica-Bold", 13, 13 * 1.35, 14)

    # Intro
    y = _draw_block(c, intro_1, x, y, maxw, "Helvetica", 11.3, 11.3 * 1.35, 10)
    y = _draw_block(c, intro_2, x, y, maxw, "Helvetica-Bold", 11.3, 11.3 * 1.35, 14)

    # Collab
    y = _draw_block(c, collab_title, x, y, maxw, "Helvetica-Bold", 12.5, 12.5 * 1.2, 6)
    y = _draw_block(c, collab_body, x, y, maxw, "Helvetica", 11.3, 11.3 * 1.35, 4)
    y = _draw_block(c, collab_list, x, y, maxw, "Helvetica", 11.3, 11.3 * 1.35, 12)

    # Identity
    y = _draw_block(c, identity_title, x, y, maxw, "Helvetica-Bold", 12.5, 12.5 * 1.2, 6)
    y = _draw_block(c, identity_list, x, y, maxw, "Helvetica", 11.3, 11.3 * 1.35, 14)

    # Calendar title
    y = _draw_block(c, calendar_title, x, y, maxw, "Helvetica-Bold", 12.5, 12.5 * 1.2, 6)

    # Calendar rows (date bold + wrapped description)
    date_w = 0.85 * inch
    body_x = x + date_w + 0.12 * inch
    maxw_body = (W - right) - body_x

    row_font = "Helvetica"
    row_size = 10.9
    row_leading = row_size * 1.35

    for d, desc in calendar:
        if y < 0.85 * inch:
            break

        c.setFont("Helvetica-Bold", 11.2)
        c.drawString(x, y, d)

        c.setFont(row_font, row_size)
        desc_lines = _wrap_lines(desc, row_font, row_size, maxw_body)

        if desc_lines:
            c.drawString(body_x, y, desc_lines[0])

        y -= row_leading
        for ln in desc_lines[1:]:
            c.drawString(body_x, y, ln)
            y -= row_leading

        y -= 3

    c.showPage()
    c.save()

    return out_pdf_path


In [22]:
# ============================================================
# !#!#!#!#!#! RUNNING STATEMENTS #!#!#!#!#!#
# ============================================================

bg_png_path = "/Users/yerik/Desktop/NEWmagicHOUSE_animations/_0_NMH_pics/NMH_tapiz.png"
out_pdf_path = "/Users/yerik/Desktop/NEWmagicHOUSE_animations/__Neu_Magic_House_Thursday_Programming_Proposal.pdf"

_pdf_2201_nmhouse_GET_letter_proposal(
    bg_png_path=bg_png_path,
    out_pdf_path=out_pdf_path,
)


'/Users/yerik/Desktop/NEWmagicHOUSE_animations/__Neu_Magic_House_Thursday_Programming_Proposal.pdf'